# InterUni Datathon 2026 - Final Submission Notebook

This notebook records the final modelling approach for **PURESIGMA**.

**Authors**

- Nick Liang
- Zhao Zhang
- Harshvir Singh

The competition objective is binary **log loss**, so lower scores are better. This notebook is intentionally lightweight: it documents the final pipeline, reads saved artifacts, validates the final submission file, and avoids launching any Optuna searches or model training runs.

## 1. Notebook Scope

The experimental work lives in the modelling notebooks and helper scripts. This notebook is the clean handoff version:

- load the train, test, sample submission, and saved model artifacts
- show how EDA and correlation analysis motivated the feature-engineering pass (§§2–4)
- reproduce the initial modelling ladder from `models.ipynb` (§5)
- document the final ensemble architecture and calibration experiments (§§6–7)
- validate the submitted CSV and record public leaderboard context (§§8–10)

Run the **imports** and **paths/helpers** cells below before any other section.

In [1]:
# --- Imports (sections 1–7) ---
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import xgboost as xgb
from catboost import CatBoostClassifier
from IPython.display import display
from lightgbm import LGBMClassifier
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, normalized_mutual_info_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 80)
pd.set_option("display.precision", 6)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
# --- Paths and column names used throughout the notebook ---
ROOT = Path.cwd()

ID_COL = "client_id"
TARGET_COL = "default"
PREDICTION_COL = "default_probability"

TRAIN_PATH = ROOT / "train.csv"
TEST_PATH = ROOT / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "sample_submission.csv"
FINAL_SUBMISSION_PATH = ROOT / "submission_global_targeted_blend.csv"

TARGETED_CONFIG_PATH = ROOT / "global_search_targeted_best.json"
PREVIOUS_GLOBAL_CONFIG_PATH = ROOT / "global_search_best.json"
CALIBRATED_CONFIG_PATH = ROOT / "global_search_calibrated_best.json"
BLEND_BEST_PATH = ROOT / "blend_best.json"
FEATURE_ENG_BEST_PATH = ROOT / "feature_eng_joint_best.json"
OPTUNA_BEST_PATH = ROOT / "optuna_best.json"

# Modelling defaults (sections 5–7)
RANDOM_STATE = 42
CV_FOLDS = 5

In [3]:
# --- Small helpers for loading artifacts and sanity-checking submissions ---

def require_file(path: Path) -> Path:
    """Fail fast if a required file is missing — easier to debug than a pandas error later."""
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return path


def load_json(path: Path) -> dict:
    with require_file(path).open(encoding="utf-8") as file:
        return json.load(file)


def read_submission(path: Path) -> pd.DataFrame:
    submission = pd.read_csv(require_file(path))
    required_columns = [ID_COL, PREDICTION_COL]
    missing_columns = [col for col in required_columns if col not in submission.columns]
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {missing_columns}")
    return submission[required_columns].copy()


def probability_summary(name: str, values: pd.Series | np.ndarray) -> dict:
    """Quick distribution summary for predicted probabilities."""
    probs = pd.Series(values, dtype="float64")
    return {
        "name": name,
        "rows": int(probs.size),
        "mean": float(probs.mean()),
        "std": float(probs.std(ddof=0)),
        "min": float(probs.min()),
        "p05": float(probs.quantile(0.05)),
        "median": float(probs.quantile(0.50)),
        "p95": float(probs.quantile(0.95)),
        "max": float(probs.max()),
    }


def validate_submission(submission: pd.DataFrame, sample_submission: pd.DataFrame) -> dict:
    """Checks we care about before uploading a CSV to the competition."""
    return {
        "rows_match_sample": bool(len(submission) == len(sample_submission)),
        "ids_match_sample": bool(submission[ID_COL].equals(sample_submission[ID_COL])),
        "has_missing_predictions": bool(submission[PREDICTION_COL].isna().any()),
        "all_probabilities_in_bounds": bool(submission[PREDICTION_COL].between(0.0, 1.0).all()),
        "duplicate_client_ids": int(submission[ID_COL].duplicated().sum()),
    }

## 2. EDA And Data Checks

The `clean.ipynb` pass established the basic schema before modelling. The data uses one row per client, a binary `default` target in train, and a sample-submission file that defines the required test ordering.

The key cleaning outcome was deliberately simple: no missing values were present, the provided numeric encodings were already usable for tree models, and `client_id` was treated only as an identifier.

In [4]:
# Load raw competition files and confirm the basics before any modelling
train_df = pd.read_csv(require_file(TRAIN_PATH))
test_df = pd.read_csv(require_file(TEST_PATH))
sample_submission = pd.read_csv(require_file(SAMPLE_SUBMISSION_PATH))

data_summary = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train_df),
            "columns": train_df.shape[1],
            "missing_cells": int(train_df.isna().sum().sum()),
            "duplicate_client_ids": int(train_df[ID_COL].duplicated().sum()),
        },
        {
            "dataset": "test",
            "rows": len(test_df),
            "columns": test_df.shape[1],
            "missing_cells": int(test_df.isna().sum().sum()),
            "duplicate_client_ids": int(test_df[ID_COL].duplicated().sum()),
        },
    ]
)

# Constant-prior baseline — any model must beat predicting the training default rate every time
target_rate = float(train_df[TARGET_COL].mean())
naive_log_loss = float(log_loss(train_df[TARGET_COL], np.repeat(target_rate, len(train_df))))

target_summary = pd.DataFrame(
    [
        {
            "target": TARGET_COL,
            "positive_rate": target_rate,
            "positive_count": int(train_df[TARGET_COL].sum()),
            "negative_count": int((1 - train_df[TARGET_COL]).sum()),
            "constant_rate_log_loss": naive_log_loss,
        }
    ]
)

display(data_summary)
display(target_summary)

,dataset,rows,columns,missing_cells,duplicate_client_ids
0,train,24000,25,0,0
1,test,6000,24,0,0


,target,positive_rate,positive_count,negative_count,constant_rate_log_loss
0,default,0.221208,5309,18691,0.528433


## 3. Correlation Analysis

Before creating new features, we inspected how the original numeric columns related to `default`. The aim was not to pick the final feature set directly, but to identify which parts of the credit history deserved the most feature-engineering attention.

Two complementary measures were used:

- **Spearman correlation** for monotonic relationships with default risk
- **normalized mutual information** for non-linear dependence after binning continuous variables

The raw-column analysis highlighted the repayment-status variables first, especially recent `PAY_*` columns. Payment amounts, bill amounts, and credit limit appeared as secondary signals, so the feature-engineering pass focused on making those histories more explicit.

In [5]:
# Rank raw columns by monotonic (Spearman) and non-linear (NMI) association with default

def discretize_for_nmi(values: pd.Series, n_bins: int = 10) -> pd.Series:
    """Bin continuous values so NMI can pick up non-linear patterns."""
    clean_values = values.replace([np.inf, -np.inf], np.nan)
    fill_value = clean_values.median()
    if pd.isna(fill_value):
        fill_value = 0.0
    clean_values = clean_values.fillna(fill_value)

    if clean_values.nunique(dropna=False) <= n_bins:
        return clean_values.astype(str)

    ranked_values = clean_values.rank(method="first")
    return pd.qcut(ranked_values, q=n_bins, duplicates="drop", labels=False).astype(str)


def correlation_feature_ranking(df: pd.DataFrame, target_col: str, top_n: int = 15) -> pd.DataFrame:
    numeric_cols = [
        col for col in df.select_dtypes(include="number").columns if col != target_col
    ]
    numeric_df = df[numeric_cols + [target_col]].replace([np.inf, -np.inf], np.nan)

    spearman = numeric_df.corr(method="spearman")[target_col].drop(target_col).abs()
    nmi_scores = pd.Series(
        {
            col: normalized_mutual_info_score(
                discretize_for_nmi(numeric_df[col]),
                df[target_col].astype(str),
            )
            for col in numeric_cols
        },
        name="nmi_with_default",
    )

    ranking = pd.concat(
        [spearman.rename("abs_spearman_with_default"), nmi_scores],
        axis=1,
    )
    ranking["mean_rank"] = ranking.rank(ascending=False).mean(axis=1)
    return ranking.sort_values("mean_rank").head(top_n)


raw_correlation_ranking = correlation_feature_ranking(train_df, TARGET_COL)
raw_correlation_ranking

,abs_spearman_with_default,nmi_with_default,mean_rank
PAY_0,0.294178,0.050355,1.0
PAY_2,0.217069,0.028158,3.0
PAY_3,0.198490,0.022206,4.0
PAY_5,0.164239,0.035626,4.0
PAY_4,0.175762,0.019504,5.0
LIMIT_BAL,0.167026,0.009996,6.0
PAY_6,0.146258,0.030720,6.0
PAY_AMT1,0.153009,0.009074,7.5
PAY_AMT2,0.148619,0.008596,8.5
PAY_AMT3,0.132164,0.006807,10.0


## 4. Feature Engineering From Correlation Signals

After the correlation analysis, feature engineering concentrated on the strongest raw signal families:

- repayment-status history from the `PAY_*` columns
- recent and severe delinquency behavior
- payment volume from `PAY_AMT*`
- bill movement from `BILL_AMT*`
- credit exposure through `LIMIT_BAL` and utilization ratios

The first-pass engineered features below came from `clean.ipynb`. They convert month-by-month history into model-friendly summaries such as total payments, total bills, credit utilization, bill trend, delay counts, severe-delay counts, and average repayment status.



| Variable | Data type | Calculation | Brief description |
|---|---|---|---|
| `total_pay` | Numeric / continuous | `PAY_AMT1 + ... + PAY_AMT6` | Total amount paid across the six observed months. |
| `total_bill` | Numeric / continuous | `BILL_AMT1 + ... + BILL_AMT6` | Total billed balance across the six observed months. |
| `credit_util_1` ... `credit_util_6` | Numeric / continuous | `BILL_AMTi / LIMIT_BAL` | Monthly credit utilisation relative to the customer's credit limit. |
| `bill_slope` | Numeric / continuous | Least-squares slope of `BILL_AMT1` ... `BILL_AMT6` over months `1,...,6` | Summarises the overall trend in bill balances across the six months. |
| `bill_abs_change_1_2` ... `bill_abs_change_5_6` | Numeric / continuous | `BILL_AMT(i+1) - BILL_AMTi` | Absolute month-to-month change in bill balance. |
| `bill_pct_change_1_2` ... `bill_pct_change_5_6` | Numeric / continuous | `(BILL_AMT(i+1) - BILL_AMTi) / BILL_AMTi` | Relative month-to-month change in bill balance. Zero denominators are replaced with `NaN`. |
| `bill_abs_change_1_6` | Numeric / continuous | `BILL_AMT6 - BILL_AMT1` | Absolute change in bill balance between months 1 and 6. |
| `bill_pct_change_1_6` | Numeric / continuous | `(BILL_AMT6 - BILL_AMT1) / BILL_AMT1` | Relative change in bill balance between months 1 and 6. |
| `max_delay` | Numeric / ordinal | `max(PAY_0, PAY_2, ..., PAY_6)` | Worst repayment-status value observed across the six months. |
| `num_months_delayed` | Integer / count | `sum(PAY_t > 0)` | Number of months in which the customer had a positive repayment delay. |
| `num_severe_delays` | Integer / count | `sum(PAY_t >= 2)` | Number of months in which the customer was at least two months behind on repayment. |
| `ever_delayed` | Binary integer | `1` if any `PAY_t > 0`, otherwise `0` | Indicates whether the customer experienced any repayment delay during the observed period. |
| `mean_pay_status` | Numeric / ordinal summary | `mean(PAY_0, PAY_2, ..., PAY_6)` | Average repayment-status value across the six months, summarising overall delinquency behaviour. |

In [6]:
# First-pass feature engineering — turn month-by-month history into summary variables
PAY_STATUS_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_AMOUNT_COLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAYMENT_AMOUNT_COLS = [f"PAY_AMT{i}" for i in range(1, 7)]


def make_first_pass_features(df: pd.DataFrame) -> pd.DataFrame:
    """Features from clean.ipynb — same logic we used to guide the later block search."""
    features = df.copy()

    features["total_pay"] = features[PAYMENT_AMOUNT_COLS].sum(axis=1)
    features["total_bill"] = features[BILL_AMOUNT_COLS].sum(axis=1)

    for month, bill_col in enumerate(BILL_AMOUNT_COLS, start=1):
        features[f"credit_util_{month}"] = features[bill_col] / features["LIMIT_BAL"]

    months = np.arange(1, 7)
    month_offsets = months - months.mean()
    bill_values = features[BILL_AMOUNT_COLS].to_numpy()
    features["bill_slope"] = (bill_values * month_offsets).sum(axis=1) / (month_offsets**2).sum()

    for month in range(1, 6):
        current_bill = f"BILL_AMT{month}"
        next_bill = f"BILL_AMT{month + 1}"
        features[f"bill_abs_change_{month}_{month + 1}"] = (
            features[next_bill] - features[current_bill]
        )
        features[f"bill_pct_change_{month}_{month + 1}"] = (
            features[next_bill] - features[current_bill]
        ) / features[current_bill].replace(0, np.nan)

    features["bill_abs_change_1_6"] = features["BILL_AMT6"] - features["BILL_AMT1"]
    features["bill_pct_change_1_6"] = (
        features["BILL_AMT6"] - features["BILL_AMT1"]
    ) / features["BILL_AMT1"].replace(0, np.nan)

    features["max_delay"] = features[PAY_STATUS_COLS].max(axis=1)
    features["num_months_delayed"] = (features[PAY_STATUS_COLS] > 0).sum(axis=1)
    features["num_severe_delays"] = (features[PAY_STATUS_COLS] >= 2).sum(axis=1)
    features["ever_delayed"] = (features[PAY_STATUS_COLS] > 0).any(axis=1).astype(int)
    features["mean_pay_status"] = features[PAY_STATUS_COLS].mean(axis=1)

    return features


analysis_df = make_first_pass_features(train_df)
print(f"Original train shape: {train_df.shape}")
print(f"First-pass feature shape: {analysis_df.shape}")

Original train shape: (24000, 25)
First-pass feature shape: (24000, 51)


In [7]:
# Re-run correlation ranking on engineered features — delay summaries should jump to the top
engineered_correlation_ranking = correlation_feature_ranking(analysis_df, TARGET_COL)
engineered_correlation_ranking

,abs_spearman_with_default,nmi_with_default,mean_rank
num_severe_delays,0.390202,0.094366,1.5
ever_delayed,0.353592,0.102319,2.0
num_months_delayed,0.388466,0.089085,2.5
PAY_0,0.294178,0.050355,4.5
max_delay,0.321376,0.044595,5.0
mean_pay_status,0.258100,0.045747,5.5
PAY_2,0.217069,0.028158,8.0
PAY_3,0.198490,0.022206,9.0
PAY_5,0.164239,0.035626,9.5
PAY_4,0.175762,0.019504,10.0


In [8]:
# Map correlation findings to the feature blocks we tested in later Optuna runs
feature_focus = pd.DataFrame(
    [
        {
            "correlation signal": "Delay severity and frequency",
            "example features": "num_severe_delays, num_months_delayed, ever_delayed, max_delay",
            "model feature blocks": "delay_engineered, pay_status, delay_trends",
            "reason for focus": "These were the strongest monotonic signals after the first feature pass.",
        },
        {
            "correlation signal": "Most recent repayment status",
            "example features": "PAY_0, PAY_2, PAY_3, mean_pay_status",
            "model feature blocks": "pay_status, delay_trends",
            "reason for focus": "Recent delinquency carried more signal than older months.",
        },
        {
            "correlation signal": "Credit exposure and repayment capacity",
            "example features": "LIMIT_BAL, total_pay, PAY_AMT1-6",
            "model feature blocks": "demographics, pay_amounts, models_copy_new_engineered",
            "reason for focus": "Credit limit and repayment volume helped separate risk levels beyond delay counts.",
        },
        {
            "correlation signal": "Bill trajectory and utilization",
            "example features": "total_bill, bill_slope, credit_util_1-6",
            "model feature blocks": "bill_amounts, bill_trends, credit_util, util_stats",
            "reason for focus": "Balance movement and utilization gave secondary credit-risk structure.",
        },
    ]
)

feature_focus

,correlation signal,example features,model feature blocks,reason for focus
0,Delay severity and frequency,"num_severe_delays, num_months_delayed, ever_de...","delay_engineered, pay_status, delay_trends",These were the strongest monotonic signals aft...
1,Most recent repayment status,"PAY_0, PAY_2, PAY_3, mean_pay_status","pay_status, delay_trends",Recent delinquency carried more signal than ol...
2,Credit exposure and repayment capacity,"LIMIT_BAL, total_pay, PAY_AMT1-6","demographics, pay_amounts, models_copy_new_eng...",Credit limit and repayment volume helped separ...
3,Bill trajectory and utilization,"total_bill, bill_slope, credit_util_1-6","bill_amounts, bill_trends, credit_util, util_s...",Balance movement and utilization gave secondar...


The final feature-engineering search therefore started with domain-shaped blocks rather than isolated columns. Delay and repayment-status blocks were treated as core candidates, while utilization, bill trends, repayment amounts, and interaction blocks were tested through cross-validation.

The later Optuna runs confirmed the broad direction but also trimmed noise: the best pulled feature-engineering model kept delay trends and utilization statistics, while dropping the delay-utilization interaction block.

## 5. Initial Modelling Path

The first modelling pipeline used stratified 5-fold cross-validation so each fold kept a similar default/non-default balance. The progression was intentionally simple:

- **Constant-prior baseline** established the minimum model had to beat.
- **Logistic regression** checked whether the correlation-led features had a useful linear probability signal.
- **XGBoost** became the first strong model because it could use non-linear thresholds and interactions in repayment history, utilization, bill movement, and payment amounts.

The initial XGBoost result motivated the broader feature-search work. Rather than manually selecting only the highest-correlation columns, features were grouped into blocks and tested with cross-validated log loss. This preserved the EDA-driven story while still letting validation decide whether secondary features and interactions helped.

We also tested selected features based on correlation as well as 


## Results Interpretation

We noted that in both models using all the variables compared to only selected features resulted in lower log loss compared to using all variables - which suggests that there is some useful signal in all variables even though they did not have high correlaiton with the response.

XGBoost had a lower mean val log loss than logistic and a higher ROC AUC value, indicating more accurate probability predictions iwth respect to the outcomes as well as a more accurate probability rankings. 

We moved forward by gradually tuning the model to observe performance

 
 


In [ ]:
# --- Section 5: baseline vs logistic vs XGBoost on two feature sets ---

DROP_COLS = [ID_COL, TARGET_COL]
all_feature_cols = [col for col in analysis_df.columns if col not in DROP_COLS]

# Same presets used in models.ipynb — compare "everything" vs correlation-led subset
FEATURE_SETS = {
    "all_engineered": all_feature_cols,
    "selected_major": [
        "num_severe_delays",
        "num_months_delayed",
        "ever_delayed",
        "max_delay",
        "PAY_0",
        "mean_pay_status",
        "PAY_2",
        "PAY_3",
        "PAY_4",
        "total_pay",
        "LIMIT_BAL",
    ],
}

y = analysis_df[TARGET_COL]


def get_models():
    """Three-model ladder: prior baseline, linear, then tree-based."""
    return {
        "baseline": DummyClassifier(strategy="prior"),
        "logistic_regression": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=1000,
                        C=1.0,
                        solver="lbfgs",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "xgboost": xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }


def run_probability_pipeline(X, y, feature_set_name, plot_roc=True):
    """5-fold CV with log loss + AUC; returns OOF predictions for ROC plots."""
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {"log_loss": "neg_log_loss", "roc_auc": "roc_auc"}

    results = []
    roc_data = {}

    for model_name, model in get_models().items():
        cv_scores = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=-1,
        )

        val_log_loss_mean = -cv_scores["test_log_loss"].mean()
        val_roc_auc_mean = cv_scores["test_roc_auc"].mean()

        oof_prob = cross_val_predict(
            model,
            X,
            y,
            cv=cv,
            method="predict_proba",
            n_jobs=-1,
        )[:, 1]

        results.append(
            {
                "feature_set": feature_set_name,
                "model": model_name,
                "train_log_loss_mean": -cv_scores["train_log_loss"].mean(),
                "val_log_loss_mean": val_log_loss_mean,
                "val_log_loss_std": cv_scores["test_log_loss"].std(),
                "train_roc_auc_mean": cv_scores["train_roc_auc"].mean(),
                "val_roc_auc_mean": val_roc_auc_mean,
                "val_roc_auc_std": cv_scores["test_roc_auc"].std(),
            }
        )

        fpr, tpr, _ = roc_curve(y, oof_prob)
        roc_data[model_name] = (fpr, tpr, val_roc_auc_mean)

    results_df = pd.DataFrame(results)

    if plot_roc:
        fig, ax = plt.subplots(figsize=(8, 6))
        for model_name, (fpr, tpr, auc_score) in roc_data.items():
            ax.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_score:.3f})")
        ax.plot([0, 1], [0, 1], "k--", label="random")
        ax.set(xlabel="False positive rate", ylabel="True positive rate")
        ax.set_title(f"ROC curves (OOF CV) — {feature_set_name}")
        ax.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

    return results_df


# Run both feature sets — full engineered matrix vs correlation-selected subset
X_all = analysis_df[FEATURE_SETS["all_engineered"]]
results_all = run_probability_pipeline(X_all, y, "all_engineered")
display(results_all.sort_values("val_log_loss_mean"))

X_selected = analysis_df[FEATURE_SETS["selected_major"]]
results_selected = run_probability_pipeline(X_selected, y, "selected_major")
display(results_selected.sort_values("val_log_loss_mean"))

In [ ]:
# XGBoost feature importance — fit on full training data for presentation slides
TOP_N_IMPORTANCE = 15

xgb_model = get_models()["xgboost"]
xgb_model.fit(X_all, y)

importance = pd.DataFrame(
    {
        "feature": X_all.columns,
        "importance": xgb_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance["importance_pct"] = 100 * importance["importance"] / importance["importance"].sum()
top_importance = importance.head(TOP_N_IMPORTANCE).iloc[::-1]

display(importance.head(TOP_N_IMPORTANCE))

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top_importance,
    x="importance_pct",
    y="feature",
    hue="feature",
    palette="Blues_r",
    legend=False,
    ax=ax,
)
ax.set_xlabel("Gain importance (% of total)")
ax.set_ylabel("")
ax.set_title(f"Top {TOP_N_IMPORTANCE} XGBoost features (all_engineered, full-train fit)")
plt.tight_layout()
plt.show()

## 6. Final Model Architecture

Section 5 showed that XGBoost beat a constant baseline and logistic regression on first-pass engineered features. The path to the submitted model followed the pipeline in `models.ipynb` and `EXPERIMENT_LOG.md`:

1. **Per-model Optuna tuning** — XGBoost, LightGBM, and CatBoost were tuned separately on out-of-fold predictions (`models.ipynb` §§9–12).
2. **Three-model convex blend** — Optuna searched blend weights over the three tuned boosters (`models.ipynb` §§13–15, saved in `blend_best.json`).
3. **Feature-engineering joint search** — block-level Optuna in `feature_eng.ipynb` produced `feature_eng_joint_best.json` (54-feature winner).
4. **Global blend search** — `run_global_search.py` pooled saved trials plus new XGB/LGB searches into a 17-candidate weighted blend (`global_search_targeted_best.json`).

The final architecture layers are:

- **Feature layer** — engineered credit-risk blocks motivated by sections 3–4.
- **Base model layer** — many Optuna-tuned XGB/LGB/CatBoost candidates, each with its own feature set.
- **OOF prediction layer** — stratified 5-fold out-of-fold probabilities per candidate.
- **Meta layer** — constrained weight vector optimized for log loss (weights sum to 1).
- **Output** — full-train refits + blend weights → `submission_global_targeted_blend.csv`.

Post-processing alternatives (logit calibration, logistic stack) are compared in section 7. The raw blend won locally and was selected for submission.

In [10]:
# Compare saved ensemble artifacts — targeted winner vs earlier global search and calibrated variant
targeted_config = load_json(TARGETED_CONFIG_PATH)
previous_global_config = load_json(PREVIOUS_GLOBAL_CONFIG_PATH) if PREVIOUS_GLOBAL_CONFIG_PATH.exists() else None

result_rows = [
    {
        "artifact": "targeted global blend",
        "source_file": TARGETED_CONFIG_PATH.name,
        "local_cv_log_loss": targeted_config["blend"]["val_log_loss_mean"],
        "local_cv_auc": targeted_config["blend"]["val_roc_auc_mean"],
        "selected": targeted_config.get("selected_submission") == "blend",
    }
]

if targeted_config.get("calibrated_blend"):
    result_rows.append(
        {
            "artifact": "targeted calibrated blend",
            "source_file": TARGETED_CONFIG_PATH.name,
            "local_cv_log_loss": targeted_config["calibrated_blend"]["val_log_loss_mean"],
            "local_cv_auc": targeted_config["calibrated_blend"]["val_roc_auc_mean"],
            "selected": targeted_config.get("selected_submission") == "calibrated_blend",
        }
    )

if previous_global_config:
    result_rows.append(
        {
            "artifact": "previous global blend",
            "source_file": PREVIOUS_GLOBAL_CONFIG_PATH.name,
            "local_cv_log_loss": previous_global_config["blend"]["val_log_loss_mean"],
            "local_cv_auc": previous_global_config["blend"]["val_roc_auc_mean"],
            "selected": False,
        }
    )

model_result_summary = pd.DataFrame(result_rows).sort_values("local_cv_log_loss")
model_result_summary

,artifact,source_file,local_cv_log_loss,local_cv_auc,selected
0,targeted global blend,global_search_targeted_best.json,0.422861,0.790076,True
1,targeted calibrated blend,global_search_targeted_best.json,0.422940,0.789938,False
2,previous global blend,global_search_best.json,0.422985,0.789890,False


In [11]:
# Final blend composition — 17 candidates; CatBoost + XGB trials dominate the weight
blend_weights = (
    pd.Series(targeted_config["blend"]["weights"], name="weight")
    .rename_axis("model_label")
    .reset_index()
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)
blend_weights["weight_pct"] = 100.0 * blend_weights["weight"]

print(f"Number of blended candidates: {len(blend_weights)}")
print(f"Weight sum: {blend_weights['weight'].sum():.6f}")
display(blend_weights.head(15))

# Bar chart for presentation — top contributors only
top_blend = blend_weights.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=top_blend,
    x="weight_pct",
    y="model_label",
    hue="model_label",
    palette="viridis",
    legend=False,
    ax=ax,
)
ax.set_xlabel("Blend weight (%)")
ax.set_ylabel("")
ax.set_title("Top 10 candidates in final global blend")
plt.tight_layout()
plt.show()

Number of blended candidates: 17
Weight sum: 1.000000


,model_label,weight,weight_pct
0,optuna_original_catboost_trial_17_all_engineered,0.193763,19.376328
1,global_search_xgboost_trial_42_exported_featur...,0.188081,18.808146
2,global_search_xgboost_trial_41_exported_featur...,0.186552,18.655245
3,optuna_extended_catboost_trial_28_all_engineer...,0.130339,13.033925
4,global_search_lightgbm_trial_41_all_features,0.077645,7.764533
5,optuna_original_lightgbm_trial_12_all_engineered,0.042849,4.284947
6,global_search_xgboost_trial_43_exported_featur...,0.041193,4.119272
7,global_search_lightgbm_trial_44_all_features,0.039707,3.970748
8,feature_eng_joint_xgboost_trial_saved_feature_...,0.035260,3.525957
9,global_search_lightgbm_trial_37_all_features,0.023435,2.343498


In [ ]:
# --- models.ipynb §11: load per-model Optuna bests (saved to optuna_best.json) ---

def load_optuna_best(path: Path = OPTUNA_BEST_PATH) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path.name} — run models.ipynb §§9–11 first.")
    return load_json(path)


MODEL_BUILDERS = {
    "xgboost": lambda params: xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    ),
    "lightgbm": lambda params: LGBMClassifier(
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
        **params,
    ),
    "catboost": lambda params: CatBoostClassifier(
        loss_function="Logloss",
        random_seed=RANDOM_STATE,
        verbose=0,
        **params,
    ),
}

optuna_best = load_optuna_best()
per_model_best = pd.DataFrame(
    [
        {
            "model": name,
            "trial": cfg["trial_number"],
            "cv_log_loss": cfg["val_log_loss_mean"],
            "cv_auc": cfg["val_roc_auc_mean"],
            "feature_set": cfg["feature_label"],
            "n_features": len(cfg["feature_cols"]),
        }
        for name, cfg in optuna_best.items()
    ]
).sort_values("cv_log_loss")

print("Per-model Optuna bests (models.ipynb §11):")
display(per_model_best)

# --- models.ipynb §13–15: convex blend of three tuned boosters ---

BLEND_MODELS = ["xgboost", "lightgbm", "catboost"]


def get_tuned_oof_probs(model_name, best_config, train_frame, target):
    feature_cols = best_config["feature_cols"]
    model = MODEL_BUILDERS[model_name](best_config["params"])
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_predict(
        model,
        train_frame[feature_cols],
        target,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]


def mean_cv_log_loss(y_true, y_prob, cv):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    fold_losses = [
        log_loss(y_true[val_idx], y_prob[val_idx]) for _, val_idx in cv.split(y_true, y_true)
    ]
    return float(np.mean(fold_losses))


def blend_weights_from_params(params):
    w_xgb = params["w_xgb"]
    w_lgb = params["w_lgb"]
    w_cat = 1.0 - w_xgb - w_lgb
    return {"xgboost": w_xgb, "lightgbm": w_lgb, "catboost": w_cat}


def blend_oof_probs(weights, oof_probs):
    return (
        weights["xgboost"] * oof_probs["xgboost"]
        + weights["lightgbm"] * oof_probs["lightgbm"]
        + weights["catboost"] * oof_probs["catboost"]
    )


def build_blend_rankings(study):
    rows = []
    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue
        weights = blend_weights_from_params(trial.params)
        rows.append(
            {
                "trial": trial.number,
                "val_log_loss_mean": trial.value,
                "w_xgboost": weights["xgboost"],
                "w_lightgbm": weights["lightgbm"],
                "w_catboost": weights["catboost"],
            }
        )
    rankings = pd.DataFrame(rows)
    if rankings.empty:
        return rankings
    return rankings.sort_values("val_log_loss_mean").reset_index(drop=True)


# Saved blend result from models.ipynb §15 (blend_best.json) — no re-run of Optuna here
early_blend = load_json(BLEND_BEST_PATH)
early_blend_weights = (
    pd.Series(early_blend["weights"], name="weight").rename_axis("model").reset_index()
)
early_blend_weights["weight_pct"] = 100 * early_blend_weights["weight"]

print("\n3-model Optuna blend result (models.ipynb §15):")
print(f"  trial: {early_blend['trial_number']}")
print(f"  CV log loss: {early_blend['val_log_loss_mean']:.6f}")
print(f"  CV AUC:      {early_blend['val_roc_auc_mean']:.6f}")
display(early_blend_weights)

In [ ]:
# --- CV progression: section 5 → models.ipynb → feature_eng → run_global_search (EXPERIMENT_LOG.md) ---

def _append_milestone(rows, stage, path, log_loss_key="blend"):
    if not path.exists():
        return
    cfg = load_json(path)
    payload = cfg.get(log_loss_key) or cfg
    rows.append(
        {
            "stage": stage,
            "artifact": path.name,
            "cv_log_loss": payload.get("val_log_loss_mean"),
            "cv_auc": payload.get("val_roc_auc_mean"),
            "n_candidates": len(payload.get("weights", cfg.get("best_feature_cols", [])))
            if log_loss_key == "blend"
            else payload.get("n_features"),
        }
    )


cv_milestones = [
    {
        "stage": "Constant-prior baseline (section 2)",
        "artifact": "—",
        "cv_log_loss": naive_log_loss,
        "cv_auc": np.nan,
        "n_candidates": 1,
    },
    {
        "stage": "3-model Optuna blend (models.ipynb §15)",
        "artifact": BLEND_BEST_PATH.name,
        "cv_log_loss": early_blend["val_log_loss_mean"],
        "cv_auc": early_blend["val_roc_auc_mean"],
        "n_candidates": len(early_blend["weights"]),
    },
]

if FEATURE_ENG_BEST_PATH.exists():
    fe_best = load_json(FEATURE_ENG_BEST_PATH)
    cv_milestones.append(
        {
            "stage": "Feature-eng joint XGB (feature_eng.ipynb)",
            "artifact": FEATURE_ENG_BEST_PATH.name,
            "cv_log_loss": fe_best["best_val_log_loss_mean"],
            "cv_auc": fe_best["best_val_roc_auc_mean"],
            "n_candidates": len(fe_best.get("best_feature_cols", [])),
        }
    )

if previous_global_config:
    _append_milestone(cv_milestones, "Global blend search (1st pass)", PREVIOUS_GLOBAL_CONFIG_PATH)
_append_milestone(cv_milestones, "Targeted global blend (final)", TARGETED_CONFIG_PATH)

cv_progression = pd.DataFrame(cv_milestones).sort_values("cv_log_loss")
print("CV log-loss progression (lower is better):")
display(cv_progression)

# Best trial per family in the final 17-candidate global-search pool
base_model_summary = pd.DataFrame(
    [
        {
            "model_family": model_name,
            "trial": cfg.get("trial_number"),
            "feature_set": cfg.get("feature_label"),
            "n_features": cfg.get("n_features"),
            "cv_log_loss": cfg.get("val_log_loss_mean"),
            "cv_auc": cfg.get("val_roc_auc_mean"),
        }
        for model_name, cfg in targeted_config.get("best_by_model", {}).items()
    ]
).sort_values("cv_log_loss")
print("\nBest trial per model family in final search pool:")
base_model_summary

## 7. Probability Calibration Experiments

Two calibration tracks were tested before locking the final blend:

**Track A — single-model nested CV** (`models copy.ipynb` §17): on the tuned XGBoost from `models.ipynb`, we compared raw OOF probabilities against Platt scaling, temperature scaling, isotonic regression, and prior shrinkage. Nested CV ensures calibrators never see the labels they are evaluated on.

**Track B — blend-level post-processing** (`run_global_search.py`): after the global blend was found, Optuna searched a logit-scale + intercept + optional prior-shrinkage layer on top of the blended OOF predictions. A logistic stack over base-model OOF streams was also tested.

In both tracks, calibration did **not** beat the raw probabilities on local log loss — the model was already well calibrated in aggregate. Section 8 shows that the same pattern held on the public leaderboard: the uncalibrated targeted blend remained best.

In [12]:
# --- models copy.ipynb §17: nested XGBoost calibration (Track A) ---

OUTER_CV_FOLDS = CV_FOLDS
INNER_CV_FOLDS = CV_FOLDS


def clip_probs(probs, eps=1e-6):
    return np.clip(probs, eps, 1 - eps)


def apply_temperature(probs, temperature):
    logits = logit(clip_probs(probs))
    return expit(logits / temperature)


def apply_shrinkage(probs, alpha, base_rate):
    return alpha * probs + (1 - alpha) * base_rate


def fit_platt(oof_probs, y_train):
    train_logits = logit(clip_probs(oof_probs)).reshape(-1, 1)
    calibrator = LogisticRegression(max_iter=1000)
    calibrator.fit(train_logits, y_train)
    return calibrator


def predict_platt(calibrator, probs):
    val_logits = logit(clip_probs(probs)).reshape(-1, 1)
    return calibrator.predict_proba(val_logits)[:, 1]


def fit_temperature(oof_probs, y_train):
    def nll(temperature):
        if temperature <= 0:
            return np.inf
        calibrated = apply_temperature(oof_probs, temperature)
        return log_loss(y_train, clip_probs(calibrated))

    result = minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded")
    return float(result.x)


def fit_shrinkage_alpha(oof_probs, y_train):
    base_rate = float(np.mean(y_train))

    def nll(alpha):
        if alpha < 0 or alpha > 1:
            return np.inf
        calibrated = apply_shrinkage(oof_probs, alpha, base_rate)
        return log_loss(y_train, clip_probs(calibrated))

    result = minimize_scalar(nll, bounds=(0.0, 1.0), method="bounded")
    return float(result.x), base_rate


def get_inner_oof_probs(X_outer_train, y_outer_train, model, inner_cv):
    inner_oof = np.zeros(len(y_outer_train))
    for inner_train_idx, inner_val_idx in inner_cv.split(X_outer_train, y_outer_train):
        fold_model = clone(model)
        fold_model.fit(
            X_outer_train.iloc[inner_train_idx],
            y_outer_train.iloc[inner_train_idx],
        )
        inner_oof[inner_val_idx] = fold_model.predict_proba(
            X_outer_train.iloc[inner_val_idx]
        )[:, 1]
    return clip_probs(inner_oof)


def fit_calibrator(method, inner_oof_probs, y_outer_train):
    if method == "raw_oof":
        return None
    if method == "platt":
        return fit_platt(inner_oof_probs, y_outer_train)
    if method == "temperature":
        return fit_temperature(inner_oof_probs, y_outer_train)
    if method == "isotonic":
        calibrator = IsotonicRegression(out_of_bounds="clip")
        calibrator.fit(inner_oof_probs, y_outer_train)
        return calibrator
    if method == "shrinkage":
        return fit_shrinkage_alpha(inner_oof_probs, y_outer_train)
    raise ValueError(f"Unknown calibration method: {method}")


def apply_calibrator(method, calibrator, probs):
    if method == "raw_oof":
        return clip_probs(probs)
    if method == "platt":
        return clip_probs(predict_platt(calibrator, probs))
    if method == "temperature":
        return clip_probs(apply_temperature(probs, calibrator))
    if method == "isotonic":
        return clip_probs(calibrator.predict(probs))
    if method == "shrinkage":
        alpha, base_rate = calibrator
        return clip_probs(apply_shrinkage(probs, alpha, base_rate))
    raise ValueError(f"Unknown calibration method: {method}")


def calibrator_summary(method, calibrator):
    if method == "raw_oof":
        return {}
    if method == "platt":
        return {
            "intercept": float(calibrator.intercept_[0]),
            "coef": float(calibrator.coef_[0, 0]),
        }
    if method == "temperature":
        return {"temperature": float(calibrator)}
    if method == "isotonic":
        return {"n_knots": len(calibrator.X_thresholds_)}
    if method == "shrinkage":
        alpha, base_rate = calibrator
        return {"alpha": float(alpha), "base_rate": float(base_rate)}
    raise ValueError(f"Unknown calibration method: {method}")


def nested_cross_calibrate_xgb_oof(X, y_target, model, method, outer_cv, inner_cv):
    oof_probs = np.zeros(len(y_target))
    fold_params = []
    for outer_train_idx, outer_val_idx in outer_cv.split(X, y_target):
        X_outer_train = X.iloc[outer_train_idx]
        y_outer_train = y_target.iloc[outer_train_idx]
        X_outer_val = X.iloc[outer_val_idx]

        inner_oof = get_inner_oof_probs(X_outer_train, y_outer_train, model, inner_cv)
        calibrator = fit_calibrator(method, inner_oof, y_outer_train)

        outer_model = clone(model)
        outer_model.fit(X_outer_train, y_outer_train)
        outer_val_raw = outer_model.predict_proba(X_outer_val)[:, 1]
        outer_val_cal = apply_calibrator(method, calibrator, outer_val_raw)

        oof_probs[outer_val_idx] = outer_val_cal
        fold_params.append(calibrator_summary(method, calibrator))
    return oof_probs, fold_params


def evaluate_calibration(name, oof_probs, y_true, cv):
    y_values = np.asarray(y_true)
    fold_log_loss = [
        log_loss(y_values[val_idx], oof_probs[val_idx]) for _, val_idx in cv.split(y_values, y_values)
    ]
    fold_roc_auc = [
        roc_auc_score(y_values[val_idx], oof_probs[val_idx]) for _, val_idx in cv.split(y_values, y_values)
    ]
    return {
        "method": name,
        "val_log_loss_mean": float(np.mean(fold_log_loss)),
        "val_log_loss_std": float(np.std(fold_log_loss)),
        "val_roc_auc_mean": float(np.mean(fold_roc_auc)),
        "val_roc_auc_std": float(np.std(fold_roc_auc)),
        "mean_pred": float(np.mean(oof_probs)),
        "base_rate": float(np.mean(y_true)),
    }


xgb_best = optuna_best["xgboost"]
X_xgb = analysis_df[xgb_best["feature_cols"]]
tuned_xgb = MODEL_BUILDERS["xgboost"](xgb_best["params"])

outer_cv = StratifiedKFold(n_splits=OUTER_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=INNER_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE + 1)

calibration_methods = ["raw_oof", "platt", "temperature", "isotonic", "shrinkage"]
calibration_results = []

for method in calibration_methods:
    oof_probs, fold_params = nested_cross_calibrate_xgb_oof(
        X_xgb, y, tuned_xgb, method, outer_cv, inner_cv
    )
    result = evaluate_calibration(method, oof_probs, y, outer_cv)
    if method == "temperature":
        result["mean_temperature"] = float(np.mean([p["temperature"] for p in fold_params]))
        result["temperature_gt_1"] = result["mean_temperature"] > 1.0
    if method == "shrinkage":
        alphas = [p["alpha"] for p in fold_params]
        result["mean_alpha"] = float(np.mean(alphas))
        result["alpha_lt_1"] = result["mean_alpha"] < 1.0
    calibration_results.append(result)

calibration_report = pd.DataFrame(calibration_results).sort_values("val_log_loss_mean")
print("Track A — nested tuned XGBoost calibration (models copy.ipynb §17):")
print(calibration_report.to_string(index=False))
print(f"\nBest calibration method: {calibration_report.iloc[0]['method']}")

# --- run_global_search.py: blend-level calibration (Track B) ---

def _add_postprocess(name, cfg, key, selected_name):
    block = cfg.get(key)
    if not block:
        return
    postprocess_rows.append(
        {
            "approach": name,
            "config_file": cfg.get("_path", "saved"),
            "cv_log_loss": block["val_log_loss_mean"],
            "cv_auc": block["val_roc_auc_mean"],
            "selected": cfg.get("selected_submission") == selected_name,
        }
    )


calibrated_config = load_json(CALIBRATED_CONFIG_PATH) if CALIBRATED_CONFIG_PATH.exists() else None
postprocess_rows = []

if calibrated_config:
    calibrated_config["_path"] = CALIBRATED_CONFIG_PATH.name
    _add_postprocess("raw blend", calibrated_config, "blend", "blend")
    _add_postprocess("calibrated blend", calibrated_config, "calibrated_blend", "calibrated_blend")
    stack = calibrated_config.get("logistic_stack")
    if stack:
        postprocess_rows.append(
            {
                "approach": "logistic stack",
                "config_file": CALIBRATED_CONFIG_PATH.name,
                "cv_log_loss": stack["val_log_loss_mean"],
                "cv_auc": stack["val_roc_auc_mean"],
                "selected": calibrated_config.get("selected_submission") == "logistic_stack",
            }
        )

targeted_config["_path"] = TARGETED_CONFIG_PATH.name
_add_postprocess("targeted raw blend", targeted_config, "blend", "blend")
_add_postprocess("targeted calibrated blend", targeted_config, "calibrated_blend", "calibrated_blend")

calibration_comparison = pd.DataFrame(postprocess_rows).sort_values("cv_log_loss")
print("\nTrack B — blend post-processing (run_global_search.py / EXPERIMENT_LOG.md):")
display(calibration_comparison)

if targeted_config.get("calibrated_blend"):
    cal_params = targeted_config["calibrated_blend"]["calibration"]
    cal_summary = pd.DataFrame(
        [
            {
                "logit_scale": cal_params["logit_scale"],
                "logit_intercept": cal_params["logit_intercept"],
                "prior_weight": cal_params["prior_weight"],
                "base_prior": targeted_config["calibrated_blend"]["base_prior"],
                "delta_log_loss_vs_raw": (
                    targeted_config["calibrated_blend"]["val_log_loss_mean"]
                    - targeted_config["blend"]["val_log_loss_mean"]
                ),
            }
        ]
    )
    print("\nTargeted calibrated-blend parameters (rejected — raw blend won):")
    display(cal_summary)

,submission,public_log_loss,delta_vs_best
0,submission_global_targeted_blend.csv,0.41217,0.00000
1,submission_global_blend.csv,0.41247,0.00030
2,submission_feature_eng.csv,0.41271,0.00054
3,submission_xgboost.csv,0.41280,0.00063
4,submission_blend.csv,0.41409,0.00192
5,submission_lightgbm.csv,0.41498,0.00281
6,submission_catboost.csv,0.41535,0.00318


## 8. Public Leaderboard Context

These public scores came from the submission dashboard after the targeted global blend was uploaded. They are external feedback only — not used for tuning. The ranking mirrors local CV: blends beat single models, and the uncalibrated targeted blend sits on top.

In [ ]:
# Public log-loss scores — confirm local CV ranking generalizes to the hold-out test set
public_scores = pd.DataFrame(
    [
        {"submission": "submission_global_targeted_blend.csv", "public_log_loss": 0.41217},
        {"submission": "submission_global_blend.csv", "public_log_loss": 0.41247},
        {"submission": "submission_feature_eng.csv", "public_log_loss": 0.41271},
        {"submission": "submission_xgboost.csv", "public_log_loss": 0.41280},
        {"submission": "submission_blend.csv", "public_log_loss": 0.41409},
        {"submission": "submission_lightgbm.csv", "public_log_loss": 0.41498},
        {"submission": "submission_catboost.csv", "public_log_loss": 0.41535},
    ]
).sort_values("public_log_loss")

public_scores["delta_vs_best"] = public_scores["public_log_loss"] - public_scores["public_log_loss"].min()
public_scores

## 9. Final Submission Validation

The selected file is `submission_global_targeted_blend.csv`. Before submitting or sharing it, validate row count, ID ordering, duplicates, missing values, and probability bounds.

In [13]:
# Sanity-check the submitted CSV against the sample submission template
final_submission = read_submission(FINAL_SUBMISSION_PATH)
validation_report = pd.DataFrame([validate_submission(final_submission, sample_submission)])
probability_report = pd.DataFrame(
    [probability_summary(FINAL_SUBMISSION_PATH.name, final_submission[PREDICTION_COL])]
)

display(validation_report)
display(probability_report)
final_submission.head()

,rows_match_sample,ids_match_sample,has_missing_predictions,all_probabilities_in_bounds,duplicate_client_ids
0,True,True,False,True,0


,name,rows,mean,std,min,p05,median,p95,max
0,submission_global_targeted_blend.csv,6000,0.218828,0.195017,0.028065,0.046959,0.14532,0.691944,0.852472


,client_id,default_probability
0,CC_0012A082B7B7,0.127502
1,CC_0012BC27DFB6,0.128202
2,CC_001563B2143D,0.111690
3,CC_001B46930B6F,0.093322
4,CC_001D771E1C9F,0.043075


In [14]:
# Compare prediction distributions across major submission files
comparison_files = [
    ROOT / "submission_global_targeted_blend.csv",
    ROOT / "submission_global_blend.csv",
    ROOT / "submission_feature_eng.csv",
    ROOT / "submission_xgboost.csv",
    ROOT / "submission_lightgbm.csv",
    ROOT / "submission_catboost.csv",
]

submission_summaries = []
for submission_path in comparison_files:
    if submission_path.exists():
        submission = read_submission(submission_path)
        submission_summaries.append(
            probability_summary(submission_path.name, submission[PREDICTION_COL])
        )

pd.DataFrame(submission_summaries).sort_values("name")

,name,rows,mean,std,min,p05,median,p95,max
5,submission_catboost.csv,6000,0.218748,0.194825,0.022347,0.049905,0.141752,0.691550,0.894570
2,submission_feature_eng.csv,6000,0.218639,0.196885,0.024210,0.044548,0.144754,0.696027,0.857077
1,submission_global_blend.csv,6000,0.218777,0.194547,0.027750,0.047897,0.145294,0.690509,0.851441
0,submission_global_targeted_blend.csv,6000,0.218828,0.195017,0.028065,0.046959,0.145320,0.691944,0.852472
4,submission_lightgbm.csv,6000,0.219429,0.195674,0.022277,0.045982,0.145960,0.696575,0.854969
3,submission_xgboost.csv,6000,0.219065,0.195657,0.021306,0.046491,0.144810,0.697403,0.830244


## 10. Reproducibility Notes

Create the local environment:

```bash
/opt/homebrew/bin/python3.13 -m venv .venv
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install -r requirements.txt
```

Rebuild the current best search artifact and submission. This is intentionally not executed in this notebook because it starts model training:

```bash
.venv/bin/python -u run_global_search.py   --trials-per-model 50   --models xgboost lightgbm catboost   --search-models xgboost lightgbm   --top-k-per-model 5   --blend-trials 1500   --calibration-trials 800   --disable-stack   --output global_search_targeted_best.json   --submission submission_global_targeted_blend.csv   --make-submission
```

The next architecture improvement should be an OOF prediction cache and model registry, so future blend searches can run without repeatedly refitting every base model.